# Bayesian Control Tower — Exploration Notebook

Use this notebook to experiment with:
- Bayesian inference mechanics (prior × likelihood → posterior)
- The `BayesianAdviceRequest` / `BayesianAdviceResponse` schemas
- Direct A2A client calls to a locally running server
- Model comparison across Anthropic / OpenAI / Gemini

In [ ]:
import sys

sys.path.insert(0, "../src")

from bayesian_control_tower.models.schemas import (
    AgentState,
    BayesianAdviceRequest,
    DecisionOption,
    DecisionPoint,
    PriorBelief,
)
from bayesian_control_tower.services.model_registry import ModelRegistry

## 1. Construct a sample request

In [ ]:
request = BayesianAdviceRequest(
    system_id="notebook-demo",
    agent_states=[
        AgentState(
            agent_id="planner-1",
            role="planner",
            current_action="awaiting_decision",
            confidence=0.65,
            observations=["task queue depth: 12", "error rate: 0.03"],
        )
    ],
    decision_point=DecisionPoint(
        description="Scale out workers or queue for next batch?",
        options=[
            DecisionOption(label="scale_out", prior_likelihood=0.4),
            DecisionOption(label="queue_batch", prior_likelihood=0.6),
        ],
    ),
    prior_beliefs=[PriorBelief(name="high_load_imminent", probability=0.55)],
    evidence=["queue_depth_rising", "recent_scale_out_succeeded"],
)

print(request.model_dump_json(indent=2))

## 2. Inspect the model registry

In [ ]:
registry = ModelRegistry.default()
for model in registry.list_models():
    print(f"{model.alias:20} → {model.litellm_model}")

## 3. Send a request to the local A2A server

Start the server first: `make run`

In [ ]:
# TODO: wire up a2a-sdk Python client once server is running
# from a2a.client import A2AClient
#
# async with A2AClient(base_url="http://localhost:8000") as client:
#     response = await client.send_task(message=request.model_dump_json())
#     print(response)

## 4. Scratch — Bayesian update by hand

Quick sanity check: manual Bayes update for the scale_out decision.

In [ ]:
# Prior
p_scale_out = 0.4
p_queue = 0.6

# P(evidence | action) — made-up likelihoods for illustration
p_evidence_given_scale = 0.8  # evidence supports scaling
p_evidence_given_queue = 0.3

# Unnormalised posteriors
unnorm_scale = p_scale_out * p_evidence_given_scale
unnorm_queue = p_queue * p_evidence_given_queue
total = unnorm_scale + unnorm_queue

posterior_scale = unnorm_scale / total
posterior_queue = unnorm_queue / total

print(f"P(scale_out | evidence) = {posterior_scale:.3f}")
print(f"P(queue     | evidence) = {posterior_queue:.3f}")